# 026-gated-correlation-head

# Imports

In [ ]:
import os

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from matplotlib.figure import Figure
from tqdm.auto import tqdm

from uavloc.data.drone_frames import DroneFrame, frame_place, load_frames
from uavloc.data.match_pairs import village_split
from uavloc.data.multidate import load_date
from uavloc.datasets import dataset_ref
from uavloc.features.factory import correlator_config
from uavloc.features.segmentation import SegmentationFeatures, load_segmenter
from uavloc.matching.benchmark import HARM_MARGIN_M, Draw, benchmark_draws, median_ms_per_correction, score_draws, summarise
from uavloc.matching.correlator import CorrelatorConfig, PositionCorrelator
from uavloc.matching.learned import LearnedScorer, build_head
from uavloc.matching.tuning import TuningGrid, tuned_config
from uavloc.track import REPO_ROOT, experiment_dir, load_runs, log_run, set_seed

# Params

In [ ]:
EXP = "026-gated-correlation-head"
NOTE = ""
LOG_RUN = True

MATCHER = "ncc"                   # "ncc" (026.1, = 025.1) | "head" (026.2 / 026.3)
HEAD_EXP = "023-learned-correlation-head"
HEAD_RUN = "023.3"                # head arm: the trained 023 run whose final head is loaded
GATE = "accept-reject"            # 022.2's gate
TEMPERATURE_GRID = [0.25, 0.5, 1.0, 2.0]
S_MAX_GRID_M = [2.0, 3.0, 5.0]
C_MIN_GRID = [0.01, 0.03, 0.05]
TUNING_NOISE_M = 10.0
MIN_APPLIED_SHARE = 0.5

PLACES = ["bushcha-50p2422-25p8305", "birky-49p9105-23p9192"]
FRAMES_DATASET = "village-crossdate-frames"
HOLDOUT_DATASET = "village-crossdate-holdout"
EXCLUDE_PLACES = ["birky-49p9105-23p9192", "bushcha-50p2422-25p8305", "dubrova-49p5175-24p0765"]
N_VAL_VILLAGES = 10
SPLIT_SEED = 42
MAP_DATE = "newest"
CLASS_WEIGHTS = (0.0, 1.0, 1.0, 0.5, 0.0, 0.5)   # background, building, track, woody, field, water: ncc and the c_min coverage
NOISE_M = [5.0, 10.0, 15.0]
PRIMARY_NOISE_M = 10.0
N_DIRECTIONS = 8
SEED = 42
DEVICE = "mps"
TORCH_THREADS = 2                 # as 025.1
SEARCH_SCALE = 1.7
FOV_W_DEG = 60.0
SAT_FEATURE_CROP_PCT = 0.03
DRONE_FEATURE_CROP_PCT = 0.0
SEG_ARCHITECTURE = "lraspp"
SEG_CHECKPOINT = "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt"
SEG_CHANNELS = "all"
MATCH_GSD_M = 0.385

TIMING_DEVICE = "cpu"
TIMING_FRAMES = 8
TIMING_REPEATS = 3

# Data

In [ ]:
set_seed(SEED)
torch.set_num_threads(TORCH_THREADS)

In [ ]:
HEAD = MATCHER == "head"
train_villages, val_villages = village_split(EXCLUDE_PLACES, N_VAL_VILLAGES, SPLIT_SEED)
len(train_villages), val_villages

In [ ]:
head_record = next(r for r in load_runs(HEAD_EXP) if r["run"] == HEAD_RUN) if HEAD else None
head_training = head_record["config"]["training"] if HEAD else None
head_training

In [ ]:
RUN_CONFIG = {
    "method": "segmentation", "matcher": MATCHER, "matching": "learned" if HEAD else "weighted-ncc",
    "class_weights": CLASS_WEIGHTS, "head_run": f"{HEAD_EXP}/{HEAD_RUN}" if HEAD else None,
    "head_seed": head_record["seed"] if HEAD else None, "head_training": head_training,
    "noise_m": NOISE_M, "primary_noise_m": PRIMARY_NOISE_M, "n_directions": N_DIRECTIONS,
    "window": "rect", "search_scale": SEARCH_SCALE, "search_radius_m": None, "prior_sigma_m": None, "gate": GATE,
    "tuning": {"temperatures": TEMPERATURE_GRID, "s_max_m": S_MAX_GRID_M, "c_min": C_MIN_GRID, "noise_m": TUNING_NOISE_M,
               "min_applied_share": MIN_APPLIED_SHARE, "scheme": "leave-one-place-out",
               "temperature_by": "mean-error", "holdout": "both primary places pooled"},
    "fov_w_deg": FOV_W_DEG, "sat_feature_crop_pct": SAT_FEATURE_CROP_PCT, "drone_feature_crop_pct": DRONE_FEATURE_CROP_PCT,
    "seg_architecture": SEG_ARCHITECTURE, "seg_checkpoint_sat": SEG_CHECKPOINT, "seg_checkpoint_drone": SEG_CHECKPOINT,
    "seg_channels": SEG_CHANNELS, "drone_seg_gsd_m": MATCH_GSD_M, "match_gsd_m": MATCH_GSD_M,
    "frames_dataset": FRAMES_DATASET, "frames_dataset_ref": dataset_ref(FRAMES_DATASET),
    "holdout_dataset": HOLDOUT_DATASET, "holdout_dataset_ref": dataset_ref(HOLDOUT_DATASET),
    "places": PLACES, "map_date": MAP_DATE,
    "split": {"exclude": EXCLUDE_PLACES, "n_val": N_VAL_VILLAGES, "seed": SPLIT_SEED, "val": val_villages},
    "torch_threads": TORCH_THREADS,
}

In [ ]:
def load_bench(dataset: str, places: list[str]) -> dict[str, list[DroneFrame]]:
    """Frames of `dataset` grouped by place; raises if a place has none."""
    frames = load_frames(dataset)
    bench = {place: [f for f in frames if frame_place(f) == place] for place in places}
    missing = [place for place, members in bench.items() if not members]
    if missing:
        raise ValueError(f"{dataset} has no frames for {missing}")
    return bench

In [ ]:
benches = {"": load_bench(FRAMES_DATASET, PLACES), "holdout_": load_bench(HOLDOUT_DATASET, val_villages)}
{prefix or "primary": {place: len(m) for place, m in bench.items()} for prefix, bench in benches.items()}

# Model

In [ ]:
model = load_segmenter(REPO_ROOT / SEG_CHECKPOINT, DEVICE, SEG_ARCHITECTURE)
base_config = correlator_config({**RUN_CONFIG, "gate": "none"})
base_config

In [ ]:
def load_head(device: str) -> torch.nn.Module | None:
    """The final head of `HEAD_RUN`, as 023 saved it."""
    if not HEAD:
        return None
    head = build_head("head", CLASS_WEIGHTS, tuple(head_training["encoder_channels"]), head_training["refine_channels"])
    state = torch.load(experiment_dir(HEAD_EXP) / "artifacts" / f"head-{HEAD_RUN}.pt", map_location="cpu")
    head.load_state_dict(state)
    return head.to(device).eval()

In [ ]:
head = load_head(DEVICE)
sum(p.numel() for p in head.parameters()) if head else 0

In [ ]:
def make_correlator(place: str, device: str, segmenter: torch.nn.Module, scorer_head: torch.nn.Module | None
                    ) -> PositionCorrelator:
    """025.1's correlator on `place`'s newest map, scored by the 023 head for the head arm."""
    features = SegmentationFeatures(segmenter, segmenter, load_date(place, MAP_DATE), SEG_CHANNELS, MATCH_GSD_M,
                                    MATCH_GSD_M, device)
    scorer = LearnedScorer(scorer_head, device) if scorer_head is not None else None
    return PositionCorrelator(features, base_config, scorer=scorer)

# Run

In [ ]:
def place_draws(place: str, members: list[DroneFrame]) -> list[Draw]:
    """025.1's draws of one place: same frames, levels, directions and seed."""
    correlator = make_correlator(place, DEVICE, model, head)
    return [d for noise_m in NOISE_M for d in benchmark_draws(correlator, members, noise_m, N_DIRECTIONS, SEED)]

In [ ]:
draws = {(prefix, place): place_draws(place, members)
         for prefix, bench in benches.items() for place, members in tqdm(bench.items(), desc=prefix or "primary")}

In [ ]:
def other_place(held_out: str) -> str:
    """Leave-one-place-out: a held-out place is scored with values fitted on the other one."""
    return next(place for place in PLACES if place != held_out)

In [ ]:
grid = TuningGrid(tuple(TEMPERATURE_GRID), tuple(S_MAX_GRID_M), tuple(C_MIN_GRID), TUNING_NOISE_M, MIN_APPLIED_SHARE)
primary_draws = {place: draws[("", place)] for place in PLACES}

In [ ]:
folds = {place: tuned_config(primary_draws[other_place(place)], base_config, GATE, grid) for place in PLACES}
holdout_fold = tuned_config([d for place in PLACES for d in primary_draws[place]], base_config, GATE, grid)
{**{place: (c.temperature, c.gate_s_max_m, c.gate_c_min) for place, (c, _) in folds.items()},
 "holdout": (holdout_fold[0].temperature, holdout_fold[0].gate_s_max_m, holdout_fold[0].gate_c_min)}

In [ ]:
def gated_config(prefix: str, place: str) -> CorrelatorConfig:
    """The primary bench scores each place with the other place's values; the holdout with values from both."""
    return folds[place][0] if prefix == "" else holdout_fold[0]

In [ ]:
def place_records(prefix: str, place: str, config: CorrelatorConfig) -> pd.DataFrame:
    records = score_draws(draws[(prefix, place)], config).assign(place=place)
    records["kept_prior"] = records["decision"] == "reject"
    records["harm"] = records["error_m"] > records["prior_error_m"] + HARM_MARGIN_M
    return records

In [ ]:
records = {prefix: pd.concat([place_records(prefix, place, gated_config(prefix, place)) for place in bench],
                             ignore_index=True) for prefix, bench in benches.items()}
records[""].groupby(["place", "noise_m"])[["error_m", "harm", "kept_prior", "reachable"]].mean().round(3)

In [ ]:
ungated = {prefix: pd.concat([place_records(prefix, place, base_config) for place in bench], ignore_index=True)
           for prefix, bench in benches.items()}
{prefix or "primary": round(r.loc[r["noise_m"] == PRIMARY_NOISE_M, "error_m"].mean(), 3) for prefix, r in ungated.items()}

In [ ]:
def blind_frames(records: pd.DataFrame) -> set[str]:
    """Frames whose query holds no hand-weighted class on any draw."""
    return set(records.groupby("frame")["coverage"].max().loc[lambda c: c == 0].index)

In [ ]:
def run_metrics(records: pd.DataFrame, bench: dict[str, list[DroneFrame]], prefix: str) -> dict[str, float]:
    """025.1's metrics plus covered / blind subsets."""
    every = {f.name for m in bench.values() for f in m}
    blind = blind_frames(records)
    subsets = {"all": every, "covered": every - blind, "blind": blind,
               **{place.split("-")[0]: {f.name for f in m} for place, m in bench.items()}}
    metrics = summarise(records, subsets, PRIMARY_NOISE_M)
    for noise_m, group in records.groupby("noise_m"):
        metrics[f"kept_prior_share_at_{noise_m:g}m"] = float(group["kept_prior"].mean())
        metrics[f"max_position_error_m_at_{noise_m:g}m"] = float(group["error_m"].max())
    primary = records[records["noise_m"] == PRIMARY_NOISE_M]
    metrics["blind_frame_share"] = len(blind) / len(every)
    for place, group in primary.groupby("place"):
        short = place.split("-")[0]
        metrics[f"accept_share_{short}"] = float((group["decision"] == "accept").mean())
        metrics[f"harm_share_{short}"] = float(group["harm"].mean())
    return {f"{prefix}{key}": value for key, value in metrics.items()}

In [ ]:
metrics = {key: value for prefix, bench in benches.items()
           for key, value in run_metrics(records[prefix], bench, prefix).items()}
metrics.update({f"ungated_{prefix}mean_position_error_m": float(r.loc[r["noise_m"] == PRIMARY_NOISE_M, "error_m"].mean())
                for prefix, r in ungated.items()})
{k: round(v, 5) for k, v in metrics.items() if "_at_" not in k and not k.startswith("holdout_")}

In [ ]:
{k: round(v, 5) for k, v in metrics.items() if "_at_" not in k and k.startswith("holdout_")}

In [ ]:
def timing_ms() -> float:
    """Median ms per correction on the CPU, drone inference and the head included."""
    segmenter = load_segmenter(REPO_ROOT / SEG_CHECKPOINT, TIMING_DEVICE, SEG_ARCHITECTURE)
    correlator = make_correlator(PLACES[0], TIMING_DEVICE, segmenter, load_head(TIMING_DEVICE))
    frames = benches[""][PLACES[0]][:TIMING_FRAMES]
    return median_ms_per_correction(correlator, frames, PRIMARY_NOISE_M, N_DIRECTIONS, TIMING_REPEATS)

In [ ]:
metrics["ms_per_correction"] = timing_ms()
metrics["ms_per_correction"]

# Plots

In [ ]:
def plot_error_cdf(gated: pd.DataFrame, raw: pd.DataFrame, title: str) -> Figure:
    """Error CDF at the primary level, gated and ungated, the level marked as a dotted line."""
    fig, ax = plt.subplots(figsize=(6, 4))
    for label, r in (("gated", gated), ("ungated", raw)):
        errors = np.sort(r.loc[r["noise_m"] == PRIMARY_NOISE_M, "error_m"].to_numpy())
        ax.plot(errors, np.linspace(0, 1, len(errors)), label=label)
    ax.axvline(PRIMARY_NOISE_M, color="k", ls=":", lw=1)
    ax.set_xlabel("position error after correction, m"); ax.set_ylabel("share of draws")
    ax.set_title(title); ax.legend(); ax.grid(alpha=.3)
    fig.tight_layout()
    return fig

In [ ]:
figures = [plot_error_cdf(records[prefix], ungated[prefix], f"{MATCHER}, {prefix or 'primary'} bench @ {PRIMARY_NOISE_M:g} m")
           for prefix in benches]

# Log

In [ ]:
for field in ("temperature", "gate_s_max_m", "gate_c_min"):
    RUN_CONFIG[field] = {place: getattr(c, field) for place, (c, _) in folds.items()}
    RUN_CONFIG[f"holdout_{field}"] = getattr(holdout_fold[0], field)

In [ ]:
artifacts = experiment_dir(EXP) / "artifacts"
artifacts.mkdir(exist_ok=True)
run_id = log_run(exp=EXP, config=RUN_CONFIG, metrics=metrics, note=NOTE, figures=figures,
                 seed=SEED, device=DEVICE)["run"] if LOG_RUN else "dry"
for prefix in benches:
    records[prefix].to_csv(artifacts / f"{prefix}records-{run_id}.csv", index=False)
    ungated[prefix].to_csv(artifacts / f"{prefix}ungated-records-{run_id}.csv", index=False)
tables = [t.assign(held_out=place) for place, (_, t) in folds.items()] + [holdout_fold[1].assign(held_out="holdout")]
pd.concat(tables, ignore_index=True).to_csv(artifacts / f"tuning-{run_id}.csv", index=False)
run_id